# 📊 IAM Ticket Analysis Dashboard
**Author:** Rudrani Mondal  
**Stack:** Python · SQL · pandas · Power BI · DAX · Matplotlib  
**Domain:** IAM & IT Operations Analytics

---

## Project Overview
This notebook performs end-to-end data quality analysis and operational validation of IT service desk (ServiceNow-style) ticket datasets. It surfaces SLA risks, recurring incident patterns, and operational KPIs — mirroring real-world IAM support analytics.

### Workflow
```
1. Synthetic Data Generation
2. Data Quality Analysis
3. Feature Engineering
4. SLA & KPI Metrics
5. Trend Analysis
6. Visualisation Dashboard
7. Power BI Export
```

## ⚙️ Cell 1 — Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# Notebook display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 120)

print('✅ All libraries imported successfully')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')

---
## 📥 Cell 2 — Synthetic Dataset Generation
Generates 500 realistic IAM support tickets with priorities, categories, SLA limits, and resolution times.

> **Note:** In a real project, replace this cell with:
> ```python
> df = pd.read_csv('servicenow_tickets.csv')   # CSV export
> # or connect via SQL:
> # import sqlite3; conn = sqlite3.connect('tickets.db'); df = pd.read_sql(query, conn)
> ```

In [ ]:
random.seed(42)
np.random.seed(42)

# ── Constants ──────────────────────────────────────────────
CATEGORIES = [
    'Access Provisioning', 'Password Reset', 'Role Assignment',
    'De-provisioning', 'Access Review', 'MFA Setup', 'Audit Request'
]
PRIORITIES  = ['P1', 'P2', 'P3', 'P4']
STATUSES    = ['Resolved', 'Closed', 'Open', 'In Progress', 'Escalated']
ASSIGNEES   = ['Team_Alpha', 'Team_Beta', 'Team_Gamma', 'Team_Delta']
SLA_LIMITS  = {'P1': 4, 'P2': 8, 'P3': 24, 'P4': 72}  # hours

# ── Generator ──────────────────────────────────────────────
def generate_tickets(n=500):
    start = datetime(2024, 1, 1)
    records = []
    for i in range(n):
        created          = start + timedelta(hours=random.randint(0, 8760))
        priority         = random.choices(PRIORITIES, weights=[5, 20, 50, 25])[0]
        sla_limit        = SLA_LIMITS[priority]
        resolution_hours = np.random.exponential(sla_limit * 0.9)
        breach           = resolution_hours > sla_limit
        resolved         = created + timedelta(hours=resolution_hours)
        status           = random.choices(STATUSES, weights=[40, 30, 10, 15, 5])[0]
        records.append({
            'ticket_id'        : f'INC{100000 + i}',
            'created_at'       : created,
            'resolved_at'      : resolved if status in ['Resolved', 'Closed'] else None,
            'priority'         : priority,
            'category'         : random.choice(CATEGORIES),
            'status'           : status,
            'assignee'         : random.choice(ASSIGNEES),
            'resolution_hours' : round(resolution_hours, 2),
            'sla_limit_hours'  : sla_limit,
            'sla_breached'     : breach,
            'reopen_count'     : random.choices([0, 1, 2], weights=[75, 20, 5])[0],
        })
    return pd.DataFrame(records)

df_raw = generate_tickets(500)

print(f'✅ Dataset generated: {df_raw.shape[0]} tickets × {df_raw.shape[1]} columns')
print(f'   Date range : {df_raw["created_at"].min().date()} → {df_raw["created_at"].max().date()}')
print(f'   SLA breaches (raw): {df_raw["sla_breached"].sum()} / {len(df_raw)}')
df_raw.head()

---
## 🔍 Cell 3 — Data Quality Analysis
Profiles completeness, uniqueness, data types, and null counts for every column.

In [ ]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a profiling report for every column in the dataframe."""
    report = []
    for col in df.columns:
        null_count  = df[col].isnull().sum()
        unique_vals = df[col].nunique()
        report.append({
            'Column'        : col,
            'DType'         : str(df[col].dtype),
            'Null Count'    : null_count,
            'Null %'        : round(null_count / len(df) * 100, 2),
            'Unique Values' : unique_vals,
            'Sample Value'  : str(df[col].dropna().iloc[0]) if null_count < len(df) else 'N/A',
        })
    return pd.DataFrame(report)

quality_report = data_quality_report(df_raw)

print('📋 DATA QUALITY REPORT')
print('=' * 75)
display(quality_report)

# Flag columns with nulls
null_cols = quality_report[quality_report['Null Count'] > 0]
if len(null_cols) > 0:
    print(f'\n⚠️  Columns with missing values: {null_cols["Column"].tolist()}')
else:
    print('\n✅ No missing values found in source columns')

---
## 🛠️ Cell 4 — Feature Engineering
Derives time-based, risk, and operational features from raw ticket data.

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Time-based features ────────────────────────────────
    df['created_hour']    = df['created_at'].dt.hour
    df['created_weekday'] = df['created_at'].dt.day_name()
    df['created_month']   = df['created_at'].dt.to_period('M').astype(str)
    df['created_quarter'] = df['created_at'].dt.to_period('Q').astype(str)
    df['is_weekend']      = df['created_at'].dt.dayofweek >= 5
    df['is_off_hours']    = (df['created_hour'] < 8) | (df['created_hour'] >= 18)

    # ── SLA features ───────────────────────────────────────
    df['sla_utilisation'] = (df['resolution_hours'] / df['sla_limit_hours']).round(3)
    df['sla_buffer_hrs']  = (df['sla_limit_hours'] - df['resolution_hours']).round(2)

    # ── Risk score (composite) ─────────────────────────────
    df['is_repeat']    = (df['reopen_count'] > 0).astype(int)
    df['risk_score']   = (
        df['sla_utilisation'].clip(0, 2) * 50 +
        df['reopen_count'] * 15 +
        df['priority'].map({'P1': 30, 'P2': 20, 'P3': 10, 'P4': 5})
    ).round(2)

    return df

df = engineer_features(df_raw)

# Show new columns
new_cols = ['ticket_id', 'priority', 'created_hour', 'created_weekday',
            'is_weekend', 'is_off_hours', 'sla_utilisation', 'sla_buffer_hrs',
            'is_repeat', 'risk_score']

print('✅ Features engineered. New columns added:')
print('   created_hour, created_weekday, created_month, created_quarter')
print('   is_weekend, is_off_hours, sla_utilisation, sla_buffer_hrs')
print('   is_repeat, risk_score')
print()
display(df[new_cols].head(8))

---
## 📈 Cell 5 — Descriptive Statistics
Summary stats for numeric and categorical columns.

In [ ]:
print('📊 NUMERIC COLUMN SUMMARY')
print('=' * 55)
display(df[['resolution_hours', 'sla_limit_hours', 'sla_utilisation',
            'risk_score', 'reopen_count']].describe().round(2))

print('\n📊 TICKET DISTRIBUTION BY CATEGORY')
print('=' * 55)
cat_dist = df['category'].value_counts().reset_index()
cat_dist.columns = ['Category', 'Count']
cat_dist['Share %'] = (cat_dist['Count'] / len(df) * 100).round(1)
display(cat_dist)

print('\n📊 TICKET DISTRIBUTION BY PRIORITY')
print('=' * 55)
pri_dist = df.groupby('priority').agg(
    Count=('ticket_id', 'count'),
    Avg_Resolution_Hrs=('resolution_hours', 'mean'),
    SLA_Breach_Count=('sla_breached', 'sum'),
    SLA_Compliance_Pct=('sla_breached', lambda x: round((1 - x.mean()) * 100, 1))
).reset_index()
display(pri_dist)

---
## 🎯 Cell 6 — KPI Computation
Computes the headline operational KPIs shown on the dashboard.

In [ ]:
def compute_kpis(df: pd.DataFrame) -> dict:
    total        = len(df)
    breached     = df['sla_breached'].sum()
    sla_rate     = round((1 - breached / total) * 100, 2)
    avg_res_hrs  = round(df['resolution_hours'].mean(), 2)
    median_res   = round(df['resolution_hours'].median(), 2)
    repeat_rate  = round(df['is_repeat'].mean() * 100, 2)
    escalated    = (df['status'] == 'Escalated').sum()
    high_risk    = (df['risk_score'] >= 80).sum()
    off_hours    = df['is_off_hours'].sum()

    kpis = {
        'Total Tickets'       : total,
        'SLA Compliance %'    : sla_rate,
        'SLA Breaches'        : int(breached),
        'Avg Resolution (hrs)': avg_res_hrs,
        'Median Resolution'   : median_res,
        'Repeat / Reopened %' : repeat_rate,
        'Escalated Tickets'   : int(escalated),
        'High Risk Tickets'   : int(high_risk),
        'Off-Hours Tickets'   : int(off_hours),
    }

    print('=' * 52)
    print('   IAM TICKET DASHBOARD — KPI SUMMARY')
    print('=' * 52)
    for k, v in kpis.items():
        val = f'{v:,.2f}' if isinstance(v, float) else f'{v:,}'
        print(f'   {k:<26} : {val}')
    print('=' * 52)
    return kpis

kpis = compute_kpis(df)

---
## 📅 Cell 7 — Trend Analysis
Monthly ticket volume and SLA breach trend.

In [ ]:
# Monthly aggregation
monthly = df.groupby('created_month').agg(
    ticket_count   = ('ticket_id',   'count'),
    sla_breaches   = ('sla_breached','sum'),
    avg_resolution = ('resolution_hours', 'mean'),
    high_risk      = ('risk_score', lambda x: (x >= 80).sum())
).reset_index()
monthly['breach_rate'] = (monthly['sla_breaches'] / monthly['ticket_count'] * 100).round(1)

print('📅 MONTHLY TREND TABLE')
print('=' * 65)
display(monthly)

# Assignee performance
print('\n👥 ASSIGNEE PERFORMANCE')
print('=' * 65)
assignee_perf = df.groupby('assignee').agg(
    Tickets          = ('ticket_id', 'count'),
    Avg_Res_Hrs      = ('resolution_hours', 'mean'),
    SLA_Compliance   = ('sla_breached', lambda x: round((1 - x.mean()) * 100, 1)),
    Avg_Risk_Score   = ('risk_score', 'mean'),
    Repeat_Tickets   = ('is_repeat', 'sum')
).reset_index()
display(assignee_perf)

# Category × Priority crosstab
print('\n🔢 SLA BREACH RATE: CATEGORY × PRIORITY')
print('=' * 65)
cross = pd.crosstab(
    df['category'], df['priority'],
    values=df['sla_breached'], aggfunc='mean'
).round(2)
display(cross)

---
## 🗃️ Cell 8 — SQL-Style Analysis with pandas
Demonstrates SQL-equivalent queries using pandas — mirrors real ServiceNow reporting workflows.

In [ ]:
# ── SQL equivalent queries using pandas ──────────────────────

# Q1: Top 10 highest-risk tickets
# SQL: SELECT ticket_id, priority, category, risk_score FROM tickets ORDER BY risk_score DESC LIMIT 10
print('🔴 Q1: TOP 10 HIGHEST-RISK TICKETS')
q1 = df.nlargest(10, 'risk_score')[['ticket_id','priority','category','risk_score','sla_breached']]
display(q1)

# Q2: SLA breach rate by category
# SQL: SELECT category, COUNT(*) AS total, SUM(sla_breached) AS breaches,
#      AVG(sla_breached)*100 AS breach_rate FROM tickets GROUP BY category ORDER BY breach_rate DESC
print('\n🔴 Q2: SLA BREACH RATE BY CATEGORY')
q2 = df.groupby('category').agg(
    Total       = ('ticket_id', 'count'),
    Breaches    = ('sla_breached', 'sum'),
    Breach_Rate = ('sla_breached', lambda x: round(x.mean() * 100, 1)),
    Avg_Risk    = ('risk_score', lambda x: round(x.mean(), 1))
).sort_values('Breach_Rate', ascending=False).reset_index()
display(q2)

# Q3: Off-hours vs business-hours SLA performance
# SQL: SELECT is_off_hours, AVG(sla_breached)*100 AS breach_rate FROM tickets GROUP BY is_off_hours
print('\n🔴 Q3: OFF-HOURS vs BUSINESS HOURS SLA COMPARISON')
q3 = df.groupby('is_off_hours').agg(
    Total       = ('ticket_id', 'count'),
    Breach_Rate = ('sla_breached', lambda x: round(x.mean() * 100, 1)),
    Avg_Res_Hrs = ('resolution_hours', lambda x: round(x.mean(), 1))
).reset_index()
q3['is_off_hours'] = q3['is_off_hours'].map({False: 'Business Hours', True: 'Off Hours'})
display(q3)

---
## 📊 Cell 9 — 6-Panel Dashboard Visualisation
Renders the full operational dashboard.

In [ ]:
# ── Plot settings ──────────────────────────────────────────
BG      = '#0F1117'
PANEL   = '#1A1D27'
ACCENT  = '#4F8EF7'
WARN    = '#F7A94F'
DANGER  = '#F74F4F'
SUCCESS = '#4FF79E'
TEXT    = 'white'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.set_title(ax.get_title(), color=TEXT, fontsize=11, pad=10)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333')

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor(BG)
fig.suptitle('IAM Ticket Analysis Dashboard — Rudrani Mondal',
             fontsize=16, fontweight='bold', color=TEXT, y=1.01)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Ticket Volume by Category ──────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cat_counts = df['category'].value_counts()
bars = ax1.barh(cat_counts.index, cat_counts.values,
                color=ACCENT, edgecolor='none', height=0.65)
for bar, val in zip(bars, cat_counts.values):
    ax1.text(bar.get_width() + 1.5, bar.get_y() + bar.get_height() / 2,
             str(val), va='center', color=TEXT, fontsize=9)
ax1.set_title('Tickets by Category')
ax1.set_xlabel('Count', color=TEXT)
ax1.tick_params(axis='y', labelcolor=TEXT)
style_ax(ax1)

# ── Panel 2: SLA Compliance by Priority ─────────────────────
ax2 = fig.add_subplot(gs[0, 1])
sla_pri = df.groupby('priority')['sla_breached'].agg(
    breached='sum', total='count'
).assign(compliance=lambda x: (1 - x.breached / x.total) * 100)
colors = [DANGER if c < 80 else WARN if c < 92 else SUCCESS
          for c in sla_pri['compliance']]
ax2.bar(sla_pri.index, sla_pri['compliance'], color=colors, edgecolor='none', width=0.55)
ax2.axhline(95, color=WARN, linestyle='--', linewidth=1.5, label='95% target')
for i, (_, row) in enumerate(sla_pri.iterrows()):
    ax2.text(i, row['compliance'] + 0.8, f"{row['compliance']:.1f}%",
             ha='center', color=TEXT, fontsize=9, fontweight='bold')
ax2.set_title('SLA Compliance by Priority (%)')
ax2.set_ylabel('%', color=TEXT)
ax2.set_ylim(0, 115)
ax2.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=9)
style_ax(ax2)

# ── Panel 3: Monthly Ticket Trend ───────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(monthly['created_month'], monthly['ticket_count'],
         color=ACCENT, marker='o', linewidth=2.5, markersize=5, label='Tickets')
ax3.fill_between(monthly['created_month'], monthly['ticket_count'],
                 alpha=0.15, color=ACCENT)
ax3b = ax3.twinx()
ax3b.plot(monthly['created_month'], monthly['breach_rate'],
          color=DANGER, marker='s', linewidth=1.8, markersize=4,
          linestyle='--', label='Breach %')
ax3b.set_ylabel('Breach Rate %', color=DANGER)
ax3b.tick_params(colors=DANGER)
ax3.set_title('Monthly Ticket Volume & Breach Rate')
ax3.set_ylabel('Tickets', color=TEXT)
ax3.tick_params(axis='x', rotation=45, labelcolor=TEXT, labelsize=8)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3b.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2,
           facecolor=PANEL, labelcolor=TEXT, fontsize=8)
style_ax(ax3)

# ── Panel 4: Resolution Time Distribution ───────────────────
ax4 = fig.add_subplot(gs[1, 0])
ax4.hist(df['resolution_hours'].clip(upper=100), bins=40,
         color=ACCENT, edgecolor=BG, alpha=0.85)
ax4.axvline(df['resolution_hours'].mean(), color=WARN, linewidth=2,
            linestyle='--', label=f"Mean: {df['resolution_hours'].mean():.1f}h")
ax4.axvline(df['resolution_hours'].median(), color=SUCCESS, linewidth=1.5,
            linestyle=':', label=f"Median: {df['resolution_hours'].median():.1f}h")
ax4.set_title('Resolution Time Distribution')
ax4.set_xlabel('Hours', color=TEXT)
ax4.set_ylabel('Frequency', color=TEXT)
ax4.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=9)
style_ax(ax4)

# ── Panel 5: SLA Breach Heatmap (Weekday × Priority) ────────
ax5 = fig.add_subplot(gs[1, 1])
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heat = df.pivot_table(
    index='created_weekday', columns='priority',
    values='sla_breached', aggfunc='mean'
)
heat = heat.reindex([d for d in day_order if d in heat.index])
im = ax5.imshow(heat.values, aspect='auto', cmap='YlOrRd')
ax5.set_xticks(range(len(heat.columns)))
ax5.set_xticklabels(heat.columns, color=TEXT, fontsize=9)
ax5.set_yticks(range(len(heat.index)))
ax5.set_yticklabels(heat.index, color=TEXT, fontsize=9)
ax5.set_title('SLA Breach Rate (Weekday × Priority)')
plt.colorbar(im, ax=ax5, label='Breach Rate')
# Annotate cells
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        val = heat.values[i, j]
        if not np.isnan(val):
            ax5.text(j, i, f'{val:.0%}', ha='center', va='center',
                     color='black' if val > 0.3 else 'white', fontsize=8)
style_ax(ax5)

# ── Panel 6: KPI Summary Card ────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
kpi_lines = [
    ('KPI SUMMARY', '', '#FFFFFF'),
    ('─' * 28, '', '#555'),
    ('Total Tickets',   f"{kpis['Total Tickets']:,}",         ACCENT),
    ('SLA Compliance',  f"{kpis['SLA Compliance %']:.1f}%",  SUCCESS),
    ('SLA Breaches',    f"{kpis['SLA Breaches']:,}",          DANGER),
    ('Avg Resolution',  f"{kpis['Avg Resolution (hrs)']:.1f} hrs", WARN),
    ('Repeat Rate',     f"{kpis['Repeat / Reopened %']:.1f}%", WARN),
    ('Escalations',     f"{kpis['Escalated Tickets']:,}",     DANGER),
    ('High-Risk Tkts',  f"{kpis['High Risk Tickets']:,}",     DANGER),
    ('Off-Hours Tkts',  f"{kpis['Off-Hours Tickets']:,}",     ACCENT),
]
y_pos = 0.95
for label, value, color in kpi_lines:
    ax6.text(0.05, y_pos, label, transform=ax6.transAxes,
             fontsize=10, color='#AAAAAA', fontfamily='monospace')
    if value:
        ax6.text(0.75, y_pos, value, transform=ax6.transAxes,
                 fontsize=10, color=color, fontfamily='monospace',
                 fontweight='bold', ha='right')
    y_pos -= 0.095
ax6.set_facecolor('#1A1D27')
ax6.patch.set_visible(True)
for spine in ax6.spines.values():
    spine.set_edgecolor('#333')

plt.savefig('iam_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Dashboard saved as iam_dashboard.png')

---
## 📊 Cell 10 — Additional Analysis: Risk Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG)

# Risk Score by Assignee
ax = axes[0]
ax.set_facecolor(PANEL)
risk_by_team = df.groupby('assignee')['risk_score'].mean().sort_values(ascending=False)
bars = ax.bar(risk_by_team.index, risk_by_team.values,
              color=[DANGER if v > 60 else WARN if v > 45 else SUCCESS for v in risk_by_team.values],
              edgecolor='none', width=0.5)
for bar, val in zip(bars, risk_by_team.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}', ha='center', color=TEXT, fontsize=9)
ax.set_title('Avg Risk Score by Assignee Team', color=TEXT, fontsize=11)
ax.set_ylabel('Risk Score', color=TEXT)
ax.axhline(50, color=WARN, linestyle='--', linewidth=1.2, label='Watch threshold (50)')
ax.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=9)
ax.tick_params(colors=TEXT)
for s in ax.spines.values(): s.set_edgecolor('#333')

# Ticket Hour-of-Day distribution
ax = axes[1]
ax.set_facecolor(PANEL)
hour_counts = df['created_hour'].value_counts().sort_index()
bar_colors  = [DANGER if h < 8 or h >= 18 else ACCENT for h in hour_counts.index]
ax.bar(hour_counts.index, hour_counts.values, color=bar_colors, edgecolor='none', width=0.85)
ax.axvspan(-0.5, 7.5,  alpha=0.08, color=DANGER, label='Off hours')
ax.axvspan(17.5, 23.5, alpha=0.08, color=DANGER)
ax.set_title('Ticket Volume by Hour of Day', color=TEXT, fontsize=11)
ax.set_xlabel('Hour (0–23)', color=TEXT)
ax.set_ylabel('Ticket Count', color=TEXT)
ax.legend(facecolor=PANEL, labelcolor=TEXT, fontsize=9)
ax.tick_params(colors=TEXT)
for s in ax.spines.values(): s.set_edgecolor('#333')

plt.tight_layout()
plt.savefig('iam_risk_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Risk analysis saved as iam_risk_analysis.png')

---
## 💾 Cell 11 — Export Clean Dataset for Power BI
Exports the enriched dataset as CSV for Power BI ingestion.

In [ ]:
# Export enriched CSV
df.to_csv('iam_tickets_clean.csv', index=False)
print(f'✅ Exported: iam_tickets_clean.csv  ({len(df)} rows × {len(df.columns)} columns)')
print(f'   Columns: {list(df.columns)}')

# Monthly summary table for Power BI
monthly.to_csv('iam_monthly_summary.csv', index=False)
print(f'✅ Exported: iam_monthly_summary.csv  ({len(monthly)} rows)')

# Assignee performance table
assignee_perf.to_csv('iam_assignee_performance.csv', index=False)
print(f'✅ Exported: iam_assignee_performance.csv')

print()
print('📌 Power BI DAX Measures to create after importing iam_tickets_clean.csv:')
print()
dax_measures = '''
SLA Compliance % =
DIVIDE(
    COUNTROWS(FILTER(Tickets, Tickets[sla_breached] = FALSE)),
    COUNTROWS(Tickets)
) * 100

Avg Resolution Hours = AVERAGE(Tickets[resolution_hours])

Breach Count = COUNTROWS(FILTER(Tickets, Tickets[sla_breached] = TRUE))

High Risk Count = COUNTROWS(FILTER(Tickets, Tickets[risk_score] >= 80))

Repeat Ticket Rate =
DIVIDE(
    COUNTROWS(FILTER(Tickets, Tickets[is_repeat] = 1)),
    COUNTROWS(Tickets)
) * 100
'''
print(dax_measures)

---
## ✅ Cell 12 — Summary & Next Steps

In [ ]:
print('=' * 60)
print('  PROJECT COMPLETE — SUMMARY')
print('=' * 60)
print()
print('  ✅ Data Quality Analysis     — null/completeness profiling')
print('  ✅ Feature Engineering       — 10 derived columns')
print('  ✅ SQL-style Analysis         — 3 analytical queries')
print('  ✅ KPI Computation            — 9 operational metrics')
print('  ✅ Trend Analysis             — monthly + assignee breakdowns')
print('  ✅ 6-Panel Dashboard          — saved as iam_dashboard.png')
print('  ✅ Risk Analysis Plots        — saved as iam_risk_analysis.png')
print('  ✅ CSV Exports (3 files)      — ready for Power BI')
print()
print('  NEXT STEPS:')
print('  → Import iam_tickets_clean.csv into Power BI')
print('  → Add DAX measures from Cell 11')
print('  → Build interactive slicers by Priority, Category, Assignee')
print('  → Connect to Project 2: SLA Breach Prediction System')
print()
print('  AUTHOR: Rudrani Mondal | rudrani.mondal05@gmail.com')
print('=' * 60)